# Data Cleaning and Preparation
## Modelling Singapore Birth and Fertility Rates (1960–2025)

**Project:** Singapore Birth and Fertility Rates Capstone

### Main tasks
1. Locate and load BirthsAndFertilityRatesAnnual.csv
2. Inspect the raw structure
3. Clean missing values and column names
4. Reshape the dataset into one row per year
5. Convert variables to numeric form
6. Check duplicates, missing values and year coverage
7. Create the training set (1960–2012)
8. Create the final test set (2013–2025)
9. Produce a data dictionary
10. Save Notebook

### Important modelling rule
The 1960–2012 period is the **training period** and 2013–2025 is the **testing period**.

In [1]:
# ============================================================
# 1. Import libraries
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# ============================================================
# 2. Locate the input CSV
# ============================================================

FILE_NAME = "BirthsAndFertilityRatesAnnual.csv"

candidate_paths = [
    f"/content/{FILE_NAME}",
    f"/mnt/data/{FILE_NAME}",
    FILE_NAME
]

DATA_PATH = next((p for p in candidate_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    matches = glob.glob(f"**/{FILE_NAME}", recursive=True)
    if matches:
        DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME}. "
        "Upload the file to Colab or place it in the notebook's working directory."
    )

print("Input file:", DATA_PATH)
print("File size:", round(os.path.getsize(DATA_PATH) / 1024, 2), "KB")

Input file: BirthsAndFertilityRatesAnnual.csv
File size: 5.98 KB


In [4]:
# ============================================================
# 3. Load the raw dataset
# ============================================================

data = pd.read_csv(DATA_PATH)

print("Shape:", data.shape)
display(data.head())

Shape: (17, 67)


,DataSeries,2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010,2009,2008,2007,2006,2005,2004,2003,2002,...,1984,1983,1982,1981,1980,1979,1978,1977,1976,1975,1974,1973,1972,1971,1970,1969,1968,1967,1966,1965,1964,1963,1962,1961,1960
0,Total Fertility Rate (TFR),0.8700,0.9700,0.9700,1.0400,1.1200,1.1000,1.1400,1.1400,1.1600,1.2000,1.2400,1.2500,1.1900,1.2900,1.2000,1.1500,1.2200,1.2800,1.2900,1.2800,1.2600,1.2600,1.2700,1.3700,...,1.6200,1.6100,1.7400,1.7800,1.8200,1.79,1.79,1.82,2.11,2.07,2.35,2.79,3.04,3.02,3.07,3.22,3.53,3.91,4.46,4.66,4.97,5.16,5.21,5.41,5.76
1,15 - 19 Years,1.3000,2.3000,2.2000,2.1000,2.2000,2.3000,2.5000,2.5000,2.6000,2.7000,2.7000,3.3000,3.8000,4.3000,4.7000,4.8000,5.0000,6.1000,6.1000,6.6000,7.2000,6.6000,6.7000,8.0000,...,10.3000,10.4000,11.4000,11.8000,12.7000,11.4,11.8,13.4,16.2,16.8,20.8,24.2,25.3,25.6,25.9,27.1,30.9,35.8,33,35.9,38.3,45.7,52,63.4,69.6
2,20 - 24 Years,8.7000,9.8000,10.6000,11.2000,11.7000,12.7000,12.7000,14.4000,15.1000,17.0000,18.7000,19.5000,19.7000,22.2000,22.4000,23.3000,25.4000,29.1000,31.2000,30.6000,32.5000,32.2000,32.4000,34.6000,...,68.6000,72.4000,80.2000,84.2000,84.9000,85.1,86.8,90.3,107.2,102.1,118.9,130.5,137.4,138.3,139,150.1,165.8,195.8,218.5,227.1,240,249,245.5,241.1,250.5
3,25 - 29 Years,38.1000,42.6000,43.7000,48.8000,53.4000,54.6000,59.4000,60.6000,62.2000,65.8000,68.7000,71.1000,70.5000,76.7000,73.4000,68.1000,74.2000,78.9000,78.7000,79.6000,80.7000,80.6000,82.2000,91.6000,...,124.2000,125.5000,136.4000,141.2000,144.5000,139.3,140.9,138.9,160.2,154.5,172.2,199.5,218,212.6,208.8,227.8,236.6,244.7,261.2,259.5,277.6,287.2,291.7,304.9,323.9
4,30 - 34 Years,69.7000,79.3000,78.7000,86.7000,92.9000,90.8000,92.4000,92.9000,93.3000,96.2000,98.5000,99.3000,90.2000,99.5000,89.5000,86.0000,90.1000,94.6000,94.4000,93.1000,89.2000,89.9000,90.0000,96.2000,...,83.0000,79.9000,84.5000,84.2000,87.8000,88.4,86.8,85.2,95.9,94.9,102.9,128.4,139.2,137.6,138,134.3,152,166.7,202,216.2,226.7,228.7,231.5,238.4,259.7


## 3. Inspect the raw dataset

The file is organised with **data series in rows** and **years in columns**.

For example, conceptually:

DataSeries | 2025 | 2024 | ... | 1960

For time-series modelling we want the final structure to be:

Year | TFR | Total_Live_Births | 

In [7]:
# ============================================================
# 4. Inspect columns and data types
# ============================================================

print("Column names:")
for i, col in enumerate(raw.columns):
    print(i, repr(col))

print("\nData types:")
display(raw.dtypes)

print("\nMissing values by column:")
display(raw.isna().sum().to_frame("missing"))

Column names:
0 'DataSeries'
1 '2025'
2 '2024'
3 '2023'
4 '2022'
5 '2021'
6 '2020'
7 '2019'
8 '2018'
9 '2017'
10 '2016'
11 '2015'
12 '2014'
13 '2013'
14 '2012'
15 '2011'
16 '2010'
17 '2009'
18 '2008'
19 '2007'
20 '2006'
21 '2005'
22 '2004'
23 '2003'
24 '2002'
25 '2001'
26 '2000'
27 '1999'
28 '1998'
29 '1997'
30 '1996'
31 '1995'
32 '1994'
33 '1993'
34 '1992'
35 '1991'
36 '1990'
37 '1989'
38 '1988'
39 '1987'
40 '1986'
41 '1985'
42 '1984'
43 '1983'
44 '1982'
45 '1981'
46 '1980'
47 '1979'
48 '1978'
49 '1977'
50 '1976'
51 '1975'
52 '1974'
53 '1973'
54 '1972'
55 '1971'
56 '1970'
57 '1969'
58 '1968'
59 '1967'
60 '1966'
61 '1965'
62 '1964'
63 '1963'
64 '1962'
65 '1961'
66 '1960'

Data types:


DataSeries        str
2025          float64
2024          float64
2023          float64
2022          float64
2021          float64
2020          float64
2019          float64
2018          float64
2017          float64
2016          float64
2015          float64
2014          float64
2013          float64
2012          float64
2011          float64
2010          float64
2009          float64
2008          float64
2007          float64
2006          float64
2005          float64
2004          float64
2003          float64
2002          float64
2001          float64
2000          float64
1999          float64
1998          float64
1997          float64
1996          float64
1995          float64
1994          float64
1993          float64
1992          float64
1991          float64
1990          float64
1989          float64
1988          float64
1987          float64
1986          float64
1985          float64
1984          float64
1983          float64
1982          float64
1981      


Missing values by column:


,missing
DataSeries,0
2025,0
2024,0
2023,0
2022,0
2021,0
2020,0
2019,0
2018,0
2017,0


In [8]:
# ============================================================
# 5. Inspect the first column / series names
# ============================================================

series_col = raw.columns[0]

print("Series-name column:", series_col)
print("\nNumber of series:", raw[series_col].nunique())

display(raw[[series_col]].head(30))

Series-name column: DataSeries

Number of series: 17


,DataSeries
0,Total Fertility Rate (TFR)
1,15 - 19 Years
2,20 - 24 Years
3,25 - 29 Years
4,30 - 34 Years
5,35 - 39 Years
6,40 - 44 Years
7,45 - 49 Years
8,Chinese
9,Malays


## 4. Standardise the raw dataset names

The original labels are useful for traceability, but they contain spaces, hyphens and punctuation. We will create modelling-friendly names such as:

- `Total_Fertility_Rate`
- `Total_Live_Births`
- `Crude_Birth_Rate`
- `Resident_Live_Births`
- `Citizen_Live_Births`

In [11]:
# ============================================================
# 6. Clean series names
# ============================================================

def clean_name(name):
    name = str(name).strip()

    # Explicitly handle the actual CSV label
    if name == "Total Fertility Rate":
        return "Total_Fertility_Rate"

    if name == "Total Live-Births":
        return "Total_Live_Births"

    if name == "Resident Live-Births":
        return "Resident_Live_Births"

    if name == "Citizen Live-Births":
        return "Citizen_Live_Births"

    if name == "Crude Birth Rate":
        return "Crude_Birth_Rate"

    if name == "Gross Reproduction Rate":
        return "Gross_Reproduction_Rate"

    if name == "Net Reproduction Rate":
        return "Net_Reproduction_Rate"

    # Age-specific fertility rates
    name = re.sub(r"^\s*(\d+)\s*-\s*(\d+)\s*Years$", r"\1_\2_Years", name)

    # General cleanup
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")

    return name


raw = raw.copy()
raw[series_col] = raw[series_col].map(clean_name)

print("Cleaned series names:")
print(raw[series_col].tolist())

Cleaned series names:
['Total_Fertility_Rate_TFR', '15_19_Years', '20_24_Years', '25_29_Years', '30_34_Years', '35_39_Years', '40_44_Years', '45_49_Years', 'Chinese', 'Malays', 'Indians', 'Gross_Reproduction_Rate', 'Net_Reproduction_Rate', 'Crude_Birth_Rate', 'Total_Live_Births', 'Resident_Live_Births', 'Citizen_Live_Births']


In [12]:
# ============================================================
# 7. Identify year columns
# ============================================================

year_columns = []

for col in raw.columns:
    try:
        year = int(str(col).strip())
        if 1960 <= year <= 2025:
            year_columns.append(col)
    except (ValueError, TypeError):
        pass

print("Number of year columns:", len(year_columns))
print("First years:", year_columns[:5])
print("Last years:", year_columns[-5:])

assert len(year_columns) == 66, (
    f"Expected 66 annual columns for 1960–2025, found {len(year_columns)}."
)

Number of year columns: 66
First years: ['2025', '2024', '2023', '2022', '2021']
Last years: ['1964', '1963', '1962', '1961', '1960']


## 6. Reshape the dataset

The raw dataset is transformed from:

**one row per indicator** into **one row per year**

In [13]:
# ============================================================
# 8. Transformed to one row per year
# ============================================================

series_names = raw[series_col].tolist()

annual = raw.set_index(series_col)[year_columns].T.reset_index()
annual = annual.rename(columns={"index": "Year"})

annual["Year"] = pd.to_numeric(annual["Year"], errors="coerce").astype("Int64")

# Convert every variable except Year to numeric.
for col in annual.columns:
    if col != "Year":
        annual[col] = (
            annual[col]
            .astype(str)
            .str.strip()
            .replace({
                "na": np.nan,
                "NA": np.nan,
                "n.a.": np.nan,
                "N.A.": np.nan,
                "-": np.nan,
                "": np.nan
            })
        )
        annual[col] = pd.to_numeric(annual[col], errors="coerce")

annual = annual.sort_values("Year").reset_index(drop=True)

print("Cleaned annual shape:", annual.shape)
display(annual.head())
display(annual.tail())

Cleaned annual shape: (66, 18)


DataSeries,Year,Total_Fertility_Rate_TFR,15_19_Years,20_24_Years,25_29_Years,30_34_Years,35_39_Years,40_44_Years,45_49_Years,Chinese,Malays,Indians,Gross_Reproduction_Rate,Net_Reproduction_Rate,Crude_Birth_Rate,Total_Live_Births,Resident_Live_Births,Citizen_Live_Births
0,1960,5.7600,69.6000,250.5000,323.9000,259.7000,176.7000,70.7000,NaN,5.6200,6.4200,7.3700,2.7800,2.5400,37.5000,"61,775.0000",NaN,NaN
1,1961,5.4100,63.4000,241.1000,304.9000,238.4000,168.9000,64.8000,NaN,5.2000,6.4200,6.9600,2.6300,2.4100,35.2000,"59,930.0000",NaN,NaN
2,1962,5.2100,52.0000,245.5000,291.7000,231.5000,156.2000,65.1000,NaN,4.9200,6.6300,6.9600,2.5300,2.3100,33.7000,"58,977.0000",NaN,NaN
3,1963,5.1600,45.7000,249.0000,287.2000,228.7000,156.1000,64.9000,NaN,4.8300,6.7500,6.9000,2.5100,2.3000,33.2000,"59,530.0000",NaN,NaN
4,1964,4.9700,38.3000,240.0000,277.6000,226.7000,147.7000,62.8000,NaN,4.6000,6.7800,7.0600,2.4200,2.2200,31.6000,"58,217.0000",NaN,NaN


DataSeries,Year,Total_Fertility_Rate_TFR,15_19_Years,20_24_Years,25_29_Years,30_34_Years,35_39_Years,40_44_Years,45_49_Years,Chinese,Malays,Indians,Gross_Reproduction_Rate,Net_Reproduction_Rate,Crude_Birth_Rate,Total_Live_Births,Resident_Live_Births,Citizen_Live_Births
61,2021,1.1200,2.2000,11.7000,53.4000,92.9000,53.6000,10.2000,0.3000,0.9600,1.8200,1.0500,0.5400,0.5400,8.6000,"38,672.0000","34,183.0000","31,713.0000"
62,2022,1.0400,2.1000,11.2000,48.8000,86.7000,49.4000,9.8000,0.4000,0.8700,1.8300,1.0100,0.5000,0.5000,7.9000,"35,605.0000","32,290.0000","30,429.0000"
63,2023,0.9700,2.2000,10.6000,43.7000,78.7000,47.9000,9.6000,0.6000,0.8100,1.6500,0.9500,0.4600,0.4600,7.4000,"33,541.0000","30,518.0000","28,877.0000"
64,2024,0.9700,2.3000,9.8000,42.6000,79.3000,50.0000,10.2000,0.7000,0.8300,1.5800,0.9100,0.4700,0.4700,7.4000,"33,703.0000","30,808.0000","29,237.0000"
65,2025,0.8700,1.3000,8.7000,38.1000,69.7000,46.0000,9.5000,0.5000,0.7000,1.5300,0.9200,0.4200,0.4200,6.5000,"29,864.0000","27,393.0000","26,071.0000"


## 7. Core variables

The project mainly focuses on:

- Total Fertility Rate
- Total Live-Births

The other variables are supporting variables

In [15]:
# ============================================================
# 9. Check core variables
# ============================================================

print("Final columns:")
print(list(annual.columns))

required_columns = [
    "Year",
    "Total_Fertility_Rate_TFR",
    "Total_Live_Births"
]

missing_required = [c for c in required_columns if c not in annual.columns]

if missing_required:
    raise ValueError(
        "Required columns were not found after cleaning: "
        + ", ".join(missing_required)
    )

print("\nCore variables found successfully:")
for c in required_columns:
    print("✓", c)

Final columns:
['Year', 'Total_Fertility_Rate_TFR', '15_19_Years', '20_24_Years', '25_29_Years', '30_34_Years', '35_39_Years', '40_44_Years', '45_49_Years', 'Chinese', 'Malays', 'Indians', 'Gross_Reproduction_Rate', 'Net_Reproduction_Rate', 'Crude_Birth_Rate', 'Total_Live_Births', 'Resident_Live_Births', 'Citizen_Live_Births']

Core variables found successfully:
✓ Year
✓ Total_Fertility_Rate_TFR
✓ Total_Live_Births


In [16]:
# ============================================================
# 10. Check minimum and maximum year and duplicates
# ============================================================

expected_years = set(range(1960, 2026))
actual_years = set(annual["Year"].dropna().astype(int))

missing_years = sorted(expected_years - actual_years)
extra_years = sorted(actual_years - expected_years)
duplicate_years = annual.loc[annual["Year"].duplicated(keep=False), "Year"].tolist()

print("Minimum year:", annual["Year"].min())
print("Maximum year:", annual["Year"].max())
print("Number of years:", annual["Year"].nunique())

print("\nMissing expected years:", missing_years)
print("Unexpected years:", extra_years)
print("Duplicate year entries:", duplicate_years)

assert not missing_years, "Some expected years are missing."
assert not extra_years, "Unexpected years were found."
assert not duplicate_years, "Duplicate years were found."

Minimum year: 1960
Maximum year: 2025
Number of years: 66

Missing expected years: []
Unexpected years: []
Duplicate year entries: []


## 8. Missing-value check

Not every series has the same historical year.

For example, some series begin in 1980 rather than 1960.

In [17]:
# ============================================================
# 11. Missing-value summary
# ============================================================

missing_summary = pd.DataFrame({
    "Variable": annual.columns,
    "Missing_Count": [annual[c].isna().sum() for c in annual.columns],
    "Non_Missing_Count": [annual[c].notna().sum() for c in annual.columns],
    "First_Available_Year": [
        annual.loc[annual[c].notna(), "Year"].min()
        if annual[c].notna().any() else np.nan
        for c in annual.columns
    ],
    "Last_Available_Year": [
        annual.loc[annual[c].notna(), "Year"].max()
        if annual[c].notna().any() else np.nan
        for c in annual.columns
    ]
})

display(missing_summary)

,Variable,Missing_Count,Non_Missing_Count,First_Available_Year,Last_Available_Year
0,Year,0,66,1960,2025
1,Total_Fertility_Rate_TFR,0,66,1960,2025
2,15_19_Years,0,66,1960,2025
3,20_24_Years,0,66,1960,2025
4,25_29_Years,0,66,1960,2025
5,30_34_Years,0,66,1960,2025
6,35_39_Years,0,66,1960,2025
7,40_44_Years,0,66,1960,2025
8,45_49_Years,20,46,1980,2025
9,Chinese,0,66,1960,2025


In [19]:
# ============================================================
# 12. Core variables check
# ============================================================

targets = ["Year","Total_Fertility_Rate_TFR", "Total_Live_Births"]

for target in targets:
    missing = annual[target].isna().sum()
    print(f"{target}: {missing} missing values")

    if missing > 0:
        print(annual.loc[annual[target].isna(), ["Year", target]])

assert annual[targets].notna().all().all(), (
    "The main targets contain missing values. "
    "Inspect the rows above before continuing."
)

Year: 0 missing values
Total_Fertility_Rate_TFR: 0 missing values
Total_Live_Births: 0 missing values


In [23]:
# ============================================================
# 13. Mean, Standard Derivation of core variable
# ============================================================

display(
    annual[["Total_Fertility_Rate_TFR", "Total_Live_Births"]]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
DataSeries,,,,,,,,
Total_Fertility_Rate_TFR,66.0000,2.0450,1.2339,0.8700,1.2600,1.6200,2.0425,5.7600
Total_Live_Births,66.0000,"44,140.0000","6,816.8061","29,864.0000","39,510.0000","42,367.0000","48,500.0000","61,775.0000"


## 10. Create the training and testing sets

These dates are fixed by the project specification.

### Training
**1960–2012**

### Final test
**2013–2025**

In [25]:
# ============================================================
# 15. Train/test split
# ============================================================

TRAIN_START = 1960
TRAIN_END = 2012
TEST_START = 2013
TEST_END = 2025

train = annual[
    (annual["Year"] >= TRAIN_START) &
    (annual["Year"] <= TRAIN_END)
].copy()

test = annual[
    (annual["Year"] >= TEST_START) &
    (annual["Year"] <= TEST_END)
].copy()

print("Training period:", train["Year"].min(), "to", train["Year"].max())
print("Training observations:", len(train))

print("\nTest period:", test["Year"].min(), "to", test["Year"].max())
print("Test observations:", len(test))

assert len(train) == 53
assert len(test) == 13

Training period: 1960 to 2012
Training observations: 53

Test period: 2013 to 2025
Test observations: 13


In [26]:
# ============================================================
# 16. Confirm no overlaps
# ============================================================

assert train["Year"].max() < test["Year"].min()

print("✓ Training and test periods do not overlap.")
print("✓ Training contains 53 annual observations.")
print("✓ Final test contains 13 annual observations.")

✓ Training and test periods do not overlap.
✓ Training contains 53 annual observations.
✓ Final test contains 13 annual observations.


## 11. Create a data dictionary

This table records what each variable represents and when it is available.

In [27]:
# ============================================================
# 17. Data dictionary
# ============================================================

data_dictionary = pd.DataFrame({
    "Variable": annual.columns,
    "Role": [
        "Time index" if c == "Year"
        else "Primary target" if c in ["Total_Fertility_Rate", "Total_Live_Births"]
        else "Supporting demographic variable"
        for c in annual.columns
    ],
    "First_Year": [
        annual.loc[annual[c].notna(), "Year"].min()
        if annual[c].notna().any() else np.nan
        for c in annual.columns
    ],
    "Last_Year": [
        annual.loc[annual[c].notna(), "Year"].max()
        if annual[c].notna().any() else np.nan
        for c in annual.columns
    ],
    "Missing_Count": [annual[c].isna().sum() for c in annual.columns]
})

display(data_dictionary)

,Variable,Role,First_Year,Last_Year,Missing_Count
0,Year,Time index,1960,2025,0
1,Total_Fertility_Rate_TFR,Supporting demographic variable,1960,2025,0
2,15_19_Years,Supporting demographic variable,1960,2025,0
3,20_24_Years,Supporting demographic variable,1960,2025,0
4,25_29_Years,Supporting demographic variable,1960,2025,0
5,30_34_Years,Supporting demographic variable,1960,2025,0
6,35_39_Years,Supporting demographic variable,1960,2025,0
7,40_44_Years,Supporting demographic variable,1960,2025,0
8,45_49_Years,Supporting demographic variable,1980,2025,20
9,Chinese,Supporting demographic variable,1960,2025,0


## 12. Save cleaned datasets

The outputs will be used for EDA.

Files created:

- `annual_birth_fertility_cleaned.csv`
- `annual_training_1960_2012.csv`
- `annual_test_2013_2025.csv`
- `data_dictionary.csv`

In [28]:
# ============================================================
# 18. Save processed datasets
# ============================================================

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

annual_path = os.path.join(OUTPUT_DIR, "annual_birth_fertility_cleaned.csv")
train_path = os.path.join(OUTPUT_DIR, "annual_training_1960_2012.csv")
test_path = os.path.join(OUTPUT_DIR, "annual_test_2013_2025.csv")
dictionary_path = os.path.join(OUTPUT_DIR, "data_dictionary.csv")

annual.to_csv(annual_path, index=False)
train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)

print("Saved:")
print("✓", annual_path)
print("✓", train_path)
print("✓", test_path)
print("✓", dictionary_path)

Saved:
✓ outputs\annual_birth_fertility_cleaned.csv
✓ outputs\annual_training_1960_2012.csv
✓ outputs\annual_test_2013_2025.csv
✓ outputs\data_dictionary.csv
